# Pipelines and ML Workflows
## EDS232, Week 9 Lab

In [31]:
# import necessary libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression

In [32]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [33]:
# read in data
url = "https://datadocs.bco-dmo.org/dataset/773466/file/B11vA82u7y2Owp/global_bleaching_environmental.csv"
df_raw = pd.read_csv(url)

/tmp/ipykernel_802667/1466987948.py:3: DtypeWarning: Columns (13,15,24) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw = pd.read_csv(url)


In [34]:
df_raw.head()

,Site_ID,Sample_ID,Data_Source,Latitude_Degrees,Longitude_Degrees,Ocean_Name,Reef_ID,Realm_Name,Ecoregion_Name,Country_Name,State_Island_Province_Name,City_Town_Name,Site_Name,Distance_to_Shore,Exposure,Turbidity,Cyclone_Frequency,Date_Day,Date_Month,Date_Year,Depth_m,Substrate_Name,Percent_Cover,Bleaching_Level,Percent_Bleaching,ClimSST,Temperature_Kelvin,Temperature_Mean,Temperature_Minimum,Temperature_Maximum,Temperature_Kelvin_Standard_Deviation,Windspeed,SSTA,SSTA_Standard_Deviation,SSTA_Mean,SSTA_Minimum,SSTA_Maximum,SSTA_Frequency,SSTA_Frequency_Standard_Deviation,SSTA_FrequencyMax,SSTA_FrequencyMean,SSTA_DHW,SSTA_DHW_Standard_Deviation,SSTA_DHWMax,SSTA_DHWMean,TSA,TSA_Standard_Deviation,TSA_Minimum,TSA_Maximum,TSA_Mean,TSA_Frequency,TSA_Frequency_Standard_Deviation,TSA_FrequencyMax,TSA_FrequencyMean,TSA_DHW,TSA_DHW_Standard_Deviation,TSA_DHWMax,TSA_DHWMean,Date,Site_Comments,Sample_Comments,Bleaching_Comments
0,2501,10324336,Donner,23.163,-82.5260,Atlantic,nd,Tropical Atlantic,Cuba and Cayman Islands,Cuba,Havana,Havana,Puntilla,8519.23,Exposed,0.0287,49.90,15,9,2005,10,nd,nd,nd,50.2,301.61,302.05,300.67,296.72,304.69,1.6,8,-0.46,1,0,-3.56,2.24,0,3.13,17,3,0,1.63,7.88,0.98,-0.8,1.6,-6.12,1.83,-2.17,0,1.09,5,0,0,0.74,7.25,0.18,2005-09-15,nd,nd,nd
1,3467,10324754,Donner,-17.575,-149.7833,Pacific,nd,Eastern Indo-Pacific,Society Islands French Polynesia,French Polynesia,Society Islands,Moorea,nd,1431.62,Exposed,0.0262,51.20,15,3,1991,14,nd,nd,nd,50.7,262.15,303.30,300.73,297.58,305.01,1.12,2,1.29,1,0,-2.73,3.1,0.5,2.77,13.25,2,0.26,1.48,11.41,0.72,1.29,1.12,-4.42,3.00,-1.26,0.25,0.93,4,0,0.26,0.67,4.65,0.19,1991-03-15,The bleaching does not appear to have gained ...,The bleaching does not appear to have gained ...,nd
2,1794,10323866,Donner,18.369,-64.5640,Atlantic,nd,Tropical Atlantic,Hispaniola Puerto Rico and Lesser Antilles,United Kingdom,British Virgin Islands,Peter Island,Coral Gardens,182.33,Exposed,0.0429,61.52,15,1,2006,7,nd,nd,nd,50.9,298.79,299.18,300.32,297.12,304.14,1.22,8,0.04,1,0,-2.92,2.83,16,4.52,23,3,0,2.45,16.24,1.26,-2.64,1.22,-4.69,2.31,-1.49,7,1.31,7,0,0,1.04,11.66,0.26,2006-01-15,nd,nd,nd
3,8647,10328028,Donner,17.760,-64.5680,Atlantic,nd,Tropical Atlantic,Hispaniola Puerto Rico and Lesser Antilles,United States,US Virgin Islands,St. Croix,3219,313.13,Exposed,0.0424,65.39,15,4,2006,9.02,nd,nd,nd,50.9,300.16,299.61,300.38,297.25,304.07,1.19,3,-0.07,1,0,-2.77,2.47,22,4.75,24,3,0,2.37,16.73,1.07,-2.27,1.19,-4.63,2.19,-1.49,3,0.94,4,0,0,0.75,5.64,0.2,2006-04-15,nd,nd,nd
4,8648,10328029,Donner,17.769,-64.5830,Atlantic,nd,Tropical Atlantic,Hispaniola Puerto Rico and Lesser Antilles,United States,US Virgin Islands,St. Croix,3194,792.0,Exposed,0.0424,65.39,15,4,2006,12.50,nd,nd,nd,50.9,300.15,299.7,300.38,296.63,303.76,1.18,3,0,1,0,-2.84,2.3,16,4.16,20,3,0,2.24,13.86,1.16,-2.19,1.18,-5.25,1.87,-1.5,3,1.33,5,0,0,0.92,6.89,0.25,2006-04-15,nd,nd,nd


## Step 1: Understand the Big Picture

**What is the response variable? Is this a regression or classification problem?**

`Percent_Bleaching` is the response variable. This is a regression problem.

**What performance metric will you use, and why is it appropriate?**

I will use MSE because the response variable is quantitative.

**What is a sensible baseline to beat (e.g., predicting the mean, a simple rule, a published benchmark)?**
Idk what this means.

**Are there any domain-specific constraints on errors? Is over-prediction or under-prediction more costly?**
If this regression is being used to predict bleaching to inform conservation efforts, then over-prediction may be more costly. We want to make sure that conservation resources are being allocated to places that are certainly at the risk of bleaching.

## Step 2: Preliminary Exploration

In [35]:
df_raw = df_raw.replace('nd', np.nan)

In [36]:
df_raw.describe(include = "all")

,Site_ID,Sample_ID,Data_Source,Latitude_Degrees,Longitude_Degrees,Ocean_Name,Reef_ID,Realm_Name,Ecoregion_Name,Country_Name,State_Island_Province_Name,City_Town_Name,Site_Name,Distance_to_Shore,Exposure,Turbidity,Cyclone_Frequency,Date_Day,Date_Month,Date_Year,Depth_m,Substrate_Name,Percent_Cover,Bleaching_Level,Percent_Bleaching,ClimSST,Temperature_Kelvin,Temperature_Mean,Temperature_Minimum,Temperature_Maximum,Temperature_Kelvin_Standard_Deviation,Windspeed,SSTA,SSTA_Standard_Deviation,SSTA_Mean,SSTA_Minimum,SSTA_Maximum,SSTA_Frequency,SSTA_Frequency_Standard_Deviation,SSTA_FrequencyMax,SSTA_FrequencyMean,SSTA_DHW,SSTA_DHW_Standard_Deviation,SSTA_DHWMax,SSTA_DHWMean,TSA,TSA_Standard_Deviation,TSA_Minimum,TSA_Maximum,TSA_Mean,TSA_Frequency,TSA_Frequency_Standard_Deviation,TSA_FrequencyMax,TSA_FrequencyMean,TSA_DHW,TSA_DHW_Standard_Deviation,TSA_DHWMax,TSA_DHWMean,Date,Site_Comments,Sample_Comments,Bleaching_Comments
count,4.136100e+04,4.136100e+04,41361,41361.000000,41361.000000,41361,28821,41361,41358,41360,41262,40228,6932,41359.00,41361,41355.0,41361.000000,41361.000000,41361.000000,41361.000000,39562,28693,28906,22531,34515,41248,41213,41229,41229,41229,41229,41232,41213,41229,41229,41185,41229,41213,41229,41229,41229,41213,41229,41229,41229,41213,41229,41229,41229,41229,41213,41229,41229,41229,41213,41229,41229,41229,41361,2257,2958,2669
unique,NaN,NaN,9,NaN,NaN,5,5354,9,114,90,480,1897,2237,13126.00,3,2387.0,NaN,NaN,NaN,NaN,472,3,359,1,2499,983,1242,813,1023,706,398,19,666,125,1,397,584,414,680,321,153,1738,591,2058,519,1126,399,921,474,608,210,465,230,76,1191,416,1705,249,5212,861,1304,7
top,NaN,NaN,Reef_Check,NaN,NaN,Pacific,151.47.039W.16.29.965S,Central Indo-Pacific,Bahamas and Florida Keys,United States,Florida,Monroe County,LL10,79.08,Sheltered,0.0,NaN,NaN,NaN,NaN,10,Hard Coral,0,Population,0,262.15,302.17,300.88,297.48,304.6,1.2,4,0,1,0,-3.31,3.02,4,6.05,23,6,0,1.57,9.01,1.56,-0.59,1.2,-4.5,2.35,-1.37,0,1.12,7,1,0,1.01,6.68,0.24,2005-08-15,% bleached = Bleaching Index,% bleached = Bleaching Index,Bleaching percentage averaged from code: Mild ...
freq,NaN,NaN,28821,NaN,NaN,21896,90,19101,4227,4611,3382,2115,48,70.00,23913,4529.0,NaN,NaN,NaN,NaN,5021,14347,6208,22531,9753,7188,211,914,539,600,1327,7470,386,6466,41229,645,535,3165,425,2389,8373,14862,477,344,572,276,1327,472,837,914,13937,722,5549,19080,28362,890,377,1373,519,157,157,1129
mean,7.455816e+04,1.012880e+07,NaN,7.558085,34.966127,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,52.159650,16.037402,6.902686,2007.796765,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,2.520418e+05,1.373151e+06,NaN,15.732185,103.404598,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.589593,7.837400,2.875063,6.073043,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,1.000000e+00,9.623000e+03,NaN,-30.262500,-179.974300,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18.310000,1.000000,1.000000,1980.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,3.502000e+03,1.031108e+07,NaN,-4.902500,-78.385600,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,47.940000,10.000000,5.000000,2003.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,5.925000e+03,1.031628e+07,NaN,10.776100,96.843300,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,50.920000,15.000000,7.000000,2007.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,8.368000e+03,1.032149e+07,NaN,20.050500,120.880400,NaN,NaN,NaN,NaN,NaN

In [37]:
df_raw.dtypes

Site_ID                                    int64
Sample_ID                                  int64
Data_Source                               object
Latitude_Degrees                         float64
Longitude_Degrees                        float64
Ocean_Name                                object
Reef_ID                                   object
Realm_Name                                object
Ecoregion_Name                            object
Country_Name                              object
State_Island_Province_Name                object
City_Town_Name                            object
Site_Name                                 object
Distance_to_Shore                         object
Exposure                                  object
Turbidity                                 object
Cyclone_Frequency                        float64
Date_Day                                   int64
Date_Month                                 int64
Date_Year                                  int64
Depth_m             

In [38]:
perc_missing = df_raw.isnull().mean() * 100

print(perc_missing)

Site_ID                                   0.000000
Sample_ID                                 0.000000
Data_Source                               0.000000
Latitude_Degrees                          0.000000
Longitude_Degrees                         0.000000
Ocean_Name                                0.000000
Reef_ID                                  30.318416
Realm_Name                                0.000000
Ecoregion_Name                            0.007253
Country_Name                              0.002418
State_Island_Province_Name                0.239356
City_Town_Name                            2.739295
Site_Name                                83.240250
Distance_to_Shore                         0.004835
Exposure                                  0.000000
Turbidity                                 0.014506
Cyclone_Frequency                         0.000000
Date_Day                                  0.000000
Date_Month                                0.000000
Date_Year                      

**How many numerical features and how many categorical features are there?**

Not all of the variables are encoded properly (some numerical features are stored as categorical), so it is unclear.

**Which columns (if any) have missing values? What percentage of observations are missing in each?**

There are no columns with missing values.

**Do the ranges in df.describe() look physically reasonable? Flag anything surprising.**

A lot of numerical vars are encoded are categorical, so it is unclear.

## Step 3: Create a representative test set and lock it away

In [39]:
# recode variables 
cols = ['Distance_to_Shore', 'Turbidity', 'Cyclone_Frequency', 'Percent_Bleaching', 'Temperature_Mean', 'Depth_m']
df_raw[cols] = df_raw[cols].apply(pd.to_numeric)

In [40]:
cols = ['Distance_to_Shore', 'Turbidity', 'Cyclone_Frequency', 'Percent_Bleaching', 'Temperature_Mean', 'Latitude_Degrees', 'Longitude_Degrees', 'Ocean_Name']

df = df_raw[cols]

In [41]:
response_var = 'Percent_Bleaching'  

X = df.drop(columns=response_var)
y = df[response_var]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training   : {X_train.shape[0]} rows')
print(f'Test       : {X_test.shape[0]} rows')

Training   : 33088 rows
Test       : 8273 rows


## Step 4: Explore the train data to gain insights

In [60]:
num_features_bleach = ['Distance_to_Shore', 'Turbidity', 'Cyclone_Frequency', 'Temperature_Mean', 'Latitude_Degrees', 'Longitude_Degrees', 'Percent_Bleaching']

In [63]:
df[num_features_bleach].corr()['Percent_Bleaching']

Distance_to_Shore    0.040749
Turbidity           -0.043618
Cyclone_Frequency    0.017477
Temperature_Mean    -0.064810
Latitude_Degrees     0.051453
Longitude_Degrees   -0.144262
Percent_Bleaching    1.000000
Name: Percent_Bleaching, dtype: float64

**Which predictors appear most strongly related to the target? You can use corr() to find out. Describe the direction.**

Longitude_Degrees is most strongly related.

**Are there too many predictors?**

"Too many predictors" can refer to too many predictors that don't contribute a lot to the model. This can be determined by referring to correlations, and removing predictors that have very low correlations with the response variable. Another method of pulling out predictors is through Lasso and ridge methods. 

In this case, I do not have too many predictors.

In [62]:
num_features = ['Distance_to_Shore', 'Turbidity', 'Cyclone_Frequency', 'Temperature_Mean', 'Latitude_Degrees', 'Longitude_Degrees']
cat_features = ['Ocean_Name']

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, num_features),
    ('cat', categorical_pipeline, cat_features)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed  = preprocessor.transform(X_test)

print(f"X_train shape after preprocessing: {X_train_processed.shape}")
print(f"X_test  shape after preprocessing: {X_test_processed.shape}")

X_train shape after preprocessing: (33088, 11)
X_test  shape after preprocessing: (8273, 11)


## Step 5: Consider Feature Combinations

## Step 6: Build Your Preprocessing Pipeline